In [68]:
AZURE_OPENAI_ENDPOINT="https://ttdevopscaedevaif-rfprfi.cognitiveservices.azure.com/"
AZURE_OPENAI_API_KEY="Fmmq2GIIXUFkhMWKSPpmekQyKM8FN2QEqKdb4FadX7XRIimz8yH0JQQJ99BHACREanaXJ3w3AAAAACOGt75K"
AZURE_OPENAI_API_VERSION="2024-12-01-preview"
AZURE_OPENAI_DEPLOYMENT_NAME="ttdevopscaedevgpt5-rfprfi"

In [69]:
from openai import AzureOpenAI

# Initialize the Azure OpenAI client
azure_openai_client = AzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,  # Adjust if your deployment uses another version
    azure_endpoint=AZURE_OPENAI_ENDPOINT
)

# Your model deployment name (check Azure portal)
DEPLOYMENT_NAME = AZURE_OPENAI_DEPLOYMENT_NAME


In [86]:
def find_additional_context(chunk_content: str,whole_document : str, prompt: str) -> str:
    
    response = azure_openai_client.chat.completions.create(
        model=DEPLOYMENT_NAME,
        messages=[ 
                {"role": "system", 
                "content": f"You are an expert context identifier specializing in multi-document RFI analysis."},
                {"role": "user", "content": f"{prompt}\n\nCHUNK_CONTENT:\n{chunk_content}\n\nWHOLE_DOCUMENT:\n{whole_document}"}
            ],
        max_completion_tokens=5000
    )
    print(response)
    return response.choices[0].message.content.strip()


In [78]:
import os
print(os.getcwd())
from processors.azure_processor import AzureDocumentProcessor #src.document_process.
from processors.file_handler import FileHandler #src.document_process.
from processors.extraction.text_extractor import TextExtractor
# =========================
#  FILE HANDLING
# =========================
text_extractor=TextExtractor()
def get_file_bytes(file_path: str) -> bytes:
    with open(file_path, "rb") as f:
        return FileHandler.process_file(f)
file_path=r"C:\Users\Harshal.chaudhari\venvs\Tetra_Tech_ZIP\tetra_Ttech_proposal_documents\five_splitted_document\705-22295407.00-B&M_HONI RFP-Chatham SS-Chatham_Lakeshore Appendix AL PCMT Works.pdf"
azure_processor = AzureDocumentProcessor()
result, client, operation_id = azure_processor.analyze_document(get_file_bytes(file_path), os.path.basename(file_path))
text_elements=text_extractor.extract_text(result)


c:\Users\Harshal.chaudhari\venvs\Tetra_Tech_ZIP\Data_indexing_code\document_processing_modular_code_26092025


In [79]:
print(text_elements)
all_text = "\n\n".join([f"[{elem.get('role', 'unknown')}] {elem['content']}" for elem in text_elements])
print(all_text)

[{'content': 'Chatham SS', 'role': <ParagraphRole.PAGE_HEADER: 'pageHeader'>, 'page_number': 2, 'paragraph_index': 5, 'position': {'x': 7.1482, 'y': 0.5447, 'page': 2}}, {'content': 'RFB Documents', 'role': <ParagraphRole.PAGE_HEADER: 'pageHeader'>, 'page_number': 2, 'paragraph_index': 6, 'position': {'x': 6.938, 'y': 1.1576, 'page': 2}}, {'content': 'FNX "INNOV', 'role': None, 'page_number': 2, 'paragraph_index': 7, 'position': {'x': 1.0285, 'y': 0.7205, 'page': 2}}, {'content': 'Appendix AL', 'role': <ParagraphRole.TITLE: 'title'>, 'page_number': 2, 'paragraph_index': 8, 'position': {'x': 0.8542, 'y': 1.6517, 'page': 2}}, {'content': '1', 'role': <ParagraphRole.PAGE_NUMBER: 'pageNumber'>, 'page_number': 2, 'paragraph_index': 28, 'position': {'x': 4.2161, 'y': 10.3656, 'page': 2}}, {'content': 'Chatham SS (NA86) Protection Description', 'role': <ParagraphRole.TITLE: 'title'>, 'page_number': 3, 'paragraph_index': 29, 'position': {'x': 1.6187, 'y': 1.1606, 'page': 3}}, {'content': 'F210

In [80]:
custom_prompt = """You are an expert at document analysis and contextualization. Your task is to provide contextual information for a text chunk to improve its searchability.

INPUTS:
- Full Document: {whole_document}
- Text Chunk: {chunk_content}

TASK:
Analyze where this chunk appears in the document and create a contextualized version that will improve search retrieval.

INSTRUCTIONS:

1. LOCATE THE CHUNK:
   - Find the exact or nearest matching location of chunk_content within whole_document
   - If exact match not found, identify the most semantically similar section

2. IDENTIFY HIERARCHICAL STRUCTURE:
   
   A. RECOGNIZE NUMBERING PATTERNS:
   Documents may use various hierarchical numbering schemes. Identify which pattern is used:
   
   - Decimal numbering: 1, 1.1, 1.2, 1.3 → 2, 2.1, 2.2, 2.3
   - Alphabetic: A, A.1, A.2 → B, B.1, B.2 or a, a.1, a.2 → b, b.1, b.2
   - Roman numerals: I, I.A, I.B → II, II.A, II.B
   - Mixed: 1, 1.a, 1.b → 2, 2.a, 2.b
   - Nested decimals: 1.1.1, 1.1.2, 1.2.1
   - Letter-number combinations: A1, A2, B1, B2
   
   B. DETERMINE PARENT-CHILD RELATIONSHIPS:
   - For decimal numbering (e.g., 1.1, 1.2, 1.3, 1.4):
     * Parent section: "Section 1"
     * Child subsections: "1.1", "1.2", "1.3", "1.4"
     * Example: If chunk is in 1.3, mention both "Section 1" and "Subsection 1.3"
   
   - For decimal numbering (e.g., 2.1, 2.2, 2.3):
     * Parent section: "Section 2"
     * Child subsections: "2.1", "2.2", "2.3"
     * Example: If chunk is in 2.1, mention both "Section 2" and "Subsection 2.1"
   
   - For alphabetic numbering (e.g., A.1, A.2, A.3):
     * Parent section: "Section A"
     * Child subsections: "A.1", "A.2", "A.3"
   
   - For deeper nesting (e.g., 1.2.3):
     * Top-level: "Section 1"
     * Mid-level: "Subsection 1.2"
     * Leaf-level: "Sub-subsection 1.2.3"
   
   C. EXTRACT SECTION NAMES AND TITLES:
   - Identify the complete section identifier (number/letter) AND its title
   - Examples:
     * "1. Introduction" → Parent: Section 1, Title: "Introduction"
     * "1.2 Background Information" → Parent: Section 1, Child: 1.2, Title: "Background Information"
     * "A. Methodology" → Parent: Section A, Title: "Methodology"
     * "2.3.1 Data Analysis" → Parent: Section 2, Mid: 2.3, Child: 2.3.1, Title: "Data Analysis"
   
   D. BUILD COMPLETE HIERARCHICAL PATH:
   Always trace from top-level to the specific subsection where the chunk appears:
   - Format: "Section [X] '[Title]' > Subsection [X.Y] '[Title]' > Sub-subsection [X.Y.Z] '[Title]'"
   - Include all levels in the hierarchy, not just the immediate section
   - If a level doesn't have a title, use only the number/letter

3. ANALYZE SURROUNDING CONTEXT:
   - What topic or concept is being discussed in this section?
   - What is the purpose of this section in the broader document?
   - Are there any key terms, definitions, or concepts introduced nearby?
   - How does this subsection relate to its parent section's theme?

4. CREATE CONTEXTUAL DESCRIPTION:
   Write 1-2 concise sentences that include:
   - The COMPLETE hierarchical path from top-level section to current subsection
   - Both the section numbers AND their descriptive titles (if available)
   - The main topic or purpose of that section
   - How this chunk relates to the broader document theme
   
   Examples:
   
   Example 1 (Decimal numbering):
   "This excerpt is from Section 2 'Methodology' > Subsection 2.3 'Data Collection Methods', which describes the research approach. It specifically addresses participant recruitment strategies used in the survey design."
   
   Example 2 (Alphabetic numbering):
   "This content appears in Section A 'Introduction' > Subsection A.2 'Research Questions', which establishes the study's objectives. It outlines the primary hypotheses tested in this research."
   
   Example 3 (Deep nesting):
   "This passage is located in Section 1 'Background' > Subsection 1.2 'Literature Review' > Sub-subsection 1.2.3 'Theoretical Frameworks', which examines existing research foundations. It discusses the cognitive load theory's application to educational settings."
   
   Example 4 (No titles, numbers only):
   "This text is from Section 3 > Subsection 3.4, which discusses implementation strategies. It covers the technical requirements for system deployment."

5. FORMAT OUTPUT:
   Combine your contextual description with the original chunk content.

   Output format:
   contextualized_chunk = "<contextual description> <chunk_content>"

IMPORTANT RULES:
- ALWAYS include the parent section when mentioning a subsection (e.g., "Section 1 > Subsection 1.3", not just "Subsection 1.3")
- Keep context description to maximum 2 sentences
- Preserve the exact chunk_content without modification
- Include section numbers/letters exactly as they appear in the document
- If section names contain special characters, include them exactly as they appear
- If no clear section structure exists, describe the thematic location (e.g., "in the discussion of X topic")
- For deeply nested sections (3+ levels), include all levels in the hierarchy
- Do not add explanations or meta-commentary, only provide the final contextualized_chunk output
- If both numbered and named sections exist, include both (e.g., "Section 2.1 'Data Analysis'")

SPECIAL CASES:
- If the document uses non-standard numbering (custom symbols, mixed formats), adapt the description accordingly but maintain the parent-child relationship
- If sections are unnumbered but have clear headings, use heading hierarchy (e.g., "Main heading 'X' > Subheading 'Y'")
- If the chunk spans multiple subsections, mention the primary subsection where it begins
"""

In [81]:
seperate_section_content = """
There are signals passed between the C60 and SEL-451-5.To maintain physical and galvanic separation between the 
relays, these signals are transmitted between the relays by using SEL 2506 CTM (contact transfer module) relays 
connected using fiber optic cables to communication between SEL-2506 and SEL-451-5. SEL-2506 will be installed in 
the ‘A’ group panel. 
The Trip module functionality is provided by both the C60 and SEL-451-5 relays. All the A group protections trip the 
breaker through C60 relay and the B group protections trip the breaker through SEL-451-5 relay. 

"""

In [85]:
summary = find_additional_context(seperate_section_content, all_text,custom_prompt)


ChatCompletion(id='chatcmpl-CMb1p23dbEA1thdikalizOhBmqCXJ', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None), content_filter_results={})], created=1759501589, model='gpt-5-2025-08-07', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=500, prompt_tokens=9326, total_tokens=9826, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=500, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=9216)), prompt_filter_results=[{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, '

In [83]:
print(summary)

In [55]:
import os
print(os.getcwd())
from processors.azure_processor import AzureDocumentProcessor #src.document_process.
from processors.file_handler import FileHandler #src.document_process.
from processors.extraction.text_extractor import TextExtractor
# =========================
#  FILE HANDLING
# =========================
text_extractor=TextExtractor()
def get_file_bytes(file_path: str) -> bytes:
    with open(file_path, "rb") as f:
        return FileHandler.process_file(f)
file_path=r"C:\Users\Harshal.chaudhari\Downloads\12092025 - renamed n xl removed nv nrg corrected\12092025 - renamed n xl removed nv nrg corrected\RFP Response\705-23895506.00-PRO-G0001-04 Solar Star 3&4 Proposal.pdf"
azure_processor = AzureDocumentProcessor()
result, client, operation_id = azure_processor.analyze_document(get_file_bytes(file_path), os.path.basename(file_path))
text_elements_1=text_extractor.extract_text(result)


c:\Users\Harshal.chaudhari\venvs\Tetra_Tech_ZIP\Data_indexing_code\document_processing_modular_code_26092025


In [56]:
all_text_1 = "\n\n".join([f"[{elem.get('role', 'unknown')}] {elem['content']}" for elem in text_elements_1])
print(all_text_1)

[None] September 7, 2023

[None] Tammy Abbey Assistant Design Phase Manager M. A. Mortenson Company Power Delivery Solutions 700 Meadow Ln N, Minneapolis, MN 55428

[None] Tammy,

[None] Tetra Tech is thankful for the invitation to be part of Solar Star 3&4 in Kern County, CA. We are pleased to submit our proposal for electrical engineering design services as outlined below and in accordance with the attachments which form an integral part of this offer.

[None] · Tetra Tech's scope of work will include engineering design and associated studies for the following scope of electrical work:

[None] - 230/34.5kV substation

[None] - Permit Package

[None] - SCE metering design, and incremental efforts to conduct a holistic power system study to include Solar Star 3 & 4 PV/BESS facility and interconnection to the existing SCE substation

[None] - SCADA RTU/Ethernet switch programming

[None] - Existing substation modification - Verify grounding grid study

[None] - Existing substation modif

In [57]:
seperate_section_content = """
Tetra Tech is thankful for the invitation to be part of Solar Star 3&4 in Kern County, CA. We are pleased to submit our proposal
for electrical engineering design services as outlined below and in accordance with the attachments which form an integral part 
of this offer. 
"""

In [87]:
summary = find_additional_context(seperate_section_content, all_text_1,custom_prompt)

ChatCompletion(id='chatcmpl-CMb5r5FE5Pp4J8ANR7IoIZwOgNXBt', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='contextualized_chunk = "This content appears under the main heading \'Tetra Tech\'s scope of work will include engineering design and associated studies for the following scope of electrical work:\' > subheading \'Existing substation modification - Protection system changes, transformer paralleling scheme\', outlining the breaker tripping architecture and inter-relay contact transfer used to maintain galvanic isolation. It situates the A- and B-group protection roles within the broader substation protection system modifications for Solar Star 3 & 4. There are signals passed between the C60 and SEL-451-5.To maintain physical and galvanic separation between the \nrelays, these signals are transmitted between the relays by using SEL 2506 CTM (contact transfer module) relays \nconnected using fiber optic cables to communication bet

In [88]:
print(summary)

contextualized_chunk = "This content appears under the main heading 'Tetra Tech's scope of work will include engineering design and associated studies for the following scope of electrical work:' > subheading 'Existing substation modification - Protection system changes, transformer paralleling scheme', outlining the breaker tripping architecture and inter-relay contact transfer used to maintain galvanic isolation. It situates the A- and B-group protection roles within the broader substation protection system modifications for Solar Star 3 & 4. There are signals passed between the C60 and SEL-451-5.To maintain physical and galvanic separation between the 
relays, these signals are transmitted between the relays by using SEL 2506 CTM (contact transfer module) relays 
connected using fiber optic cables to communication between SEL-2506 and SEL-451-5. SEL-2506 will be installed in 
the ‘A’ group panel. 
The Trip module functionality is provided by both the C60 and SEL-451-5 relays. All th

In [63]:
import os
print(os.getcwd())
from processors.azure_processor import AzureDocumentProcessor #src.document_process.
from processors.file_handler import FileHandler #src.document_process.
from processors.extraction.text_extractor import TextExtractor
# =========================
#  FILE HANDLING
# =========================
text_extractor=TextExtractor()
def get_file_bytes(file_path: str) -> bytes:
    with open(file_path, "rb") as f:
        return FileHandler.process_file(f)
file_path=r"C:\Users\Harshal.chaudhari\venvs\Tetra_Tech_ZIP\tetra_Ttech_proposal_documents\five_splitted_document\705-22295407.00-B&M_HONI RFP-Chatham SS-Chatham_Lambton Appendix A_B Planning Specs.pdf"
azure_processor = AzureDocumentProcessor()
result, client, operation_id = azure_processor.analyze_document(get_file_bytes(file_path), os.path.basename(file_path))
text_elements_2=text_extractor.extract_text(result)


c:\Users\Harshal.chaudhari\venvs\Tetra_Tech_ZIP\Data_indexing_code\document_processing_modular_code_26092025


In [64]:
all_text_2 = "\n\n".join([f"[{elem.get('role', 'unknown')}] {elem['content']}" for elem in text_elements_2])
print(all_text_2)

[ParagraphRole.TITLE] APPENDIX AJ-B Chatham SS/Lambton TS - Planning Specification

[ParagraphRole.PAGE_FOOTER] fnx-innov.com

[None] 483 Bay Street, Toronto, Ontario, M5G 2P5

[None] Planning Specification and Request for Release Estimates

[None] Build New 2-230 kV Circuit Lambton TS x Chatham SS Line

[None] Appropriation Request: 27187

[None] Prepared by:

[None] Reviewed by:

[None] Emeka Okongwu, P.Eng. Senior Network Management Engineer System Planning

[None] Mark Brodie, P.Eng. Manager - Transmission Planning System Planning

[ParagraphRole.TITLE] AR 27187 - Build New 2-230 kV Circuit Lambton TS x Chatham SS Line

[ParagraphRole.SECTION_HEADING] INTRODUCTION

[None] Two projects are currently in progress in the Windsor - Essex Region to reinforce the bulk transmission west of Chatham: AR25910 (Lakeshore TS - Build New Station), and AR 25436 (Build Chatham SS x Lakeshore TS New 2 x 230kV Line). These projects are intended to increase the overall transfer capability of the bulk

In [65]:
seperate_section_content = """
Carry out the necessary work to obtain Environmental Assessment (EA) and OEB Leave 
to Construct (Section 92) approvals for building the new double circuit line. The 
approvals shall include the proposed work at the terminal stations. Costs for carrying out 
the approvals are to be included in the line work estimate. As part of the EA process, 
determine the appropriate route for the new line. Figure 2 is a map of the area showing 
existing transmission.
"""

In [66]:
summary = find_additional_context(seperate_section_content, all_text_2,custom_prompt)

In [67]:
print(summary)

contextualized_chunk = "This excerpt is from Section 1.0 'Line Work' > Subsection 'EA Work', which outlines the environmental assessment and regulatory approvals required for the construction of the new double-circuit 230 kV transmission line between Lambton TS and Chatham SS. It details the process for obtaining Environmental Assessment (EA) and Ontario Energy Board (OEB) Leave to Construct (Section 92) approvals, including route determination and terminal station work. Carry out the necessary work to obtain Environmental Assessment (EA) and OEB Leave to Construct (Section 92) approvals for building the new double circuit line. The approvals shall include the proposed work at the terminal stations. Costs for carrying out the approvals are to be included in the line work estimate. As part of the EA process, determine the appropriate route for the new line. Figure 2 is a map of the area showing existing transmission."
